# M17 — Quantify Uncertainty

**Whole first:** inspect model probabilities, outcomes, and an action decision before studying probability notation. This notebook is a deterministic uncertainty workbench: `model outputs → repeated outcomes → reliability groups → action under consequences`.

For every experiment, write your answer at the **Prediction checkpoint** before running the next code cell. Keep predictions outside the source notebook so the repository contains no prefilled learner evidence.

In [ ]:
from collections import defaultdict
from pathlib import Path
import csv
import random

RANDOM_SEED = 1704
DATA_CANDIDATES = [
    Path.cwd() / 'datasets' / 'M17' / 'model_predictions.csv',
    Path.cwd().parent / 'datasets' / 'M17' / 'model_predictions.csv',
    Path('..') / 'datasets' / 'M17' / 'model_predictions.csv',
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Run from the repository root or labs directory.')

def load_predictions(path):
    records = []
    with path.open(encoding='utf-8', newline='') as handle:
        for raw in csv.DictReader(handle):
            records.append({
                'case_id': raw['case_id'],
                'cohort': raw['cohort'],
                'model_probability': float(raw['model_probability']),
                'outcome': int(raw['outcome']),
            })
    if not records:
        raise ValueError('The model-output fixture must not be empty.')
    if any(row['outcome'] not in (0, 1) for row in records):
        raise ValueError('Outcomes must be binary event indicators.')
    return records

rows = load_predictions(DATA_PATH)
print({'rows': len(rows), 'events': sum(row['outcome'] for row in rows), 'cohorts': sorted({row['cohort'] for row in rows})})

## Start with model outputs, not formulas

A probability is a graded claim about uncertainty, not a guarantee for one case.

**Prediction checkpoint — write before running:** Which score groups will have observed event frequencies close to their reported probabilities? Is the `deployment_shift` cohort likely to behave like the stable cohort? Name the repeated group you are using as the reference population.

In [ ]:
def event_frequency(outcomes):
    values = list(outcomes)
    if not values:
        raise ValueError('Frequency needs at least one observation.')
    return sum(values) / len(values)

def summarize_model_outputs(records):
    return {
        'cases': len(records),
        'event_count': sum(row['outcome'] for row in records),
        'base_rate': event_frequency(row['outcome'] for row in records),
        'mean_model_probability': sum(row['model_probability'] for row in records) / len(records),
    }

MODEL_SUMMARY = summarize_model_outputs(rows)
print(MODEL_SUMMARY)

**Prediction checkpoint — write before running:** If a 0.80 group is calibrated, about what fraction of repeated cases should contain the event? What would count as evidence of miscalibration for a 0.90 group?

In [ ]:
def calibration_table(records):
    grouped = defaultdict(list)
    for row in records:
        grouped[row['model_probability']].append(row)
    table = []
    for probability, group in sorted(grouped.items()):
        observed = event_frequency(row['outcome'] for row in group)
        table.append({
            'model_probability': probability,
            'count': len(group),
            'observed_frequency': observed,
            'calibration_gap': observed - probability,
            'cohorts': sorted({row['cohort'] for row in group}),
        })
    return table

CALIBRATION = calibration_table(rows)
for group in CALIBRATION:
    print(group)

Pause before formalism. Describe only what the counts show: the repeated group, number of events, observed fraction, reported probability, and difference. Do not label an individual 0 or 1 outcome as proof that its probability was right or wrong.

## Events and frequency through simulation

**Prediction checkpoint — write before running:** For an event with probability 0.30, rank 20, 200, and 20,000 trials from most to least variable in their observed frequency. Predict whether any run must equal exactly 0.30.

In [ ]:
def simulate_event_frequency(probability, trials, seed=RANDOM_SEED):
    if not 0 <= probability <= 1:
        raise ValueError('Probability must be between 0 and 1.')
    if trials <= 0:
        raise ValueError('Trials must be positive.')
    rng = random.Random(seed)
    events = sum(rng.random() < probability for _ in range(trials))
    return {'trials': trials, 'events': events, 'frequency': events / trials}

FREQUENCY_RUNS = [simulate_event_frequency(0.30, trials) for trials in (20, 200, 20_000)]
for run in FREQUENCY_RUNS:
    print(run)

Now name the structure you observed. An **event** is an outcome or set of outcomes of interest. Its **complement** is everything in the reference population where it does not occur. **Empirical frequency** is `event count / trial count`. A probability describes the long-run generating uncertainty; a finite frequency can differ because the sample is finite.

In [ ]:
def event_and_complement_counts(outcomes):
    values = list(outcomes)
    event_count = sum(values)
    return {'total': len(values), 'event': event_count, 'complement': len(values) - event_count}

MODEL_EVENT_COUNTS = event_and_complement_counts(row['outcome'] for row in rows)
print(MODEL_EVENT_COUNTS)

## Conditional probability and base rates from counts

A screening system is applied to 10,000 people. The condition base rate is 1%, sensitivity is 90%, and specificity is 91%.

**Prediction checkpoint — write before running:** Among positive results, predict whether the condition frequency will be near 1%, 50%, or 90%. Sketch affected and unaffected rows before constructing the count table.

In [ ]:
def screening_counts(population, base_rate, sensitivity, specificity):
    if population <= 0:
        raise ValueError('Population must be positive.')
    if any(not 0 <= value <= 1 for value in (base_rate, sensitivity, specificity)):
        raise ValueError('Rates must be between 0 and 1.')
    affected = round(population * base_rate)
    unaffected = population - affected
    true_positive = round(affected * sensitivity)
    false_negative = affected - true_positive
    true_negative = round(unaffected * specificity)
    false_positive = unaffected - true_negative
    return {
        'population': population,
        'affected': affected,
        'unaffected': unaffected,
        'true_positive': true_positive,
        'false_negative': false_negative,
        'false_positive': false_positive,
        'true_negative': true_negative,
        'all_positive': true_positive + false_positive,
    }

SCREENING = screening_counts(10_000, base_rate=0.01, sensitivity=0.90, specificity=0.91)
print(SCREENING)

Only after seeing the count table, introduce direction. `P(positive | condition)` uses affected people as its denominator. `P(condition | positive)` uses all positive results as its denominator. Reversing the words on either side of “given” changes the reference group. The **base rate** is the event frequency before conditioning on the result.

In [ ]:
def conditional_probability(joint_count, condition_count):
    if condition_count <= 0:
        raise ValueError('The conditioning group must not be empty.')
    if not 0 <= joint_count <= condition_count:
        raise ValueError('Joint count must belong to the conditioning group.')
    return joint_count / condition_count

P_POSITIVE_GIVEN_CONDITION = conditional_probability(SCREENING['true_positive'], SCREENING['affected'])
P_CONDITION_GIVEN_POSITIVE = conditional_probability(SCREENING['true_positive'], SCREENING['all_positive'])
print({
    'P(positive | condition)': P_POSITIVE_GIVEN_CONDITION,
    'P(condition | positive)': P_CONDITION_GIVEN_POSITIVE,
})

## Controlled failure — base-rate neglect

Failed claim: “The test is 90% sensitive, so a positive result means a 90% chance of the condition.”

**Prediction checkpoint — write before running:** Identify the denominator silently substituted by the failed claim. Predict which count in the table will make the repair most different from 90%.

In [ ]:
FAILED_BASE_RATE_CLAIM = P_POSITIVE_GIVEN_CONDITION
REPAIRED_BASE_RATE_RESULT = P_CONDITION_GIVEN_POSITIVE
BASE_RATE_NEGLECT_FACTOR = FAILED_BASE_RATE_CLAIM / REPAIRED_BASE_RATE_RESULT
print({
    'failed_claim': FAILED_BASE_RATE_CLAIM,
    'count_derived_result': REPAIRED_BASE_RATE_RESULT,
    'overstatement_factor': BASE_RATE_NEGLECT_FACTOR,
    'neglected_false_positives': SCREENING['false_positive'],
})

## Independence and dependence

In 100 operating periods, alert A appears in 20, alert B appears in 20, and both appear in 14.

**Prediction checkpoint — write before running:** If the alerts were independent, predict how many joint periods you would expect. Then compare that reference with 14 before naming the relationship.

In [ ]:
def independence_report(total, count_a, count_b, count_both, tolerance=1e-12):
    if total <= 0:
        raise ValueError('Total must be positive.')
    if not 0 <= count_both <= min(count_a, count_b) <= total:
        raise ValueError('Counts do not describe valid overlapping events.')
    p_a = count_a / total
    p_b = count_b / total
    observed_joint = count_both / total
    independent_joint = p_a * p_b
    gap = observed_joint - independent_joint
    return {
        'P(A)': p_a,
        'P(B)': p_b,
        'observed_P(A_and_B)': observed_joint,
        'independent_reference': independent_joint,
        'gap': gap,
        'relationship': 'independent' if abs(gap) <= tolerance else 'dependent',
    }

DEPENDENCE = independence_report(100, count_a=20, count_b=20, count_both=14)
print(DEPENDENCE)

Now formalize the check. Under independence, the joint probability equals the product of the marginal probabilities: `P(A and B) = P(A) × P(B)`. Dependence means learning one event changes what should be expected about the other. Similar names, shared causes, or convenient multiplication do not establish independence.

## Expected loss turns probability into a decision input

Acting now costs 12 units. Missing an event costs 100 units.

**Prediction checkpoint — write before running:** For probabilities 0.10 and 0.40, predict whether acting or waiting has lower expected loss. State whether your choice would stay the same if the miss cost changed.

In [ ]:
def expected_losses(probability, action_cost, miss_cost):
    if not 0 <= probability <= 1:
        raise ValueError('Probability must be between 0 and 1.')
    if action_cost < 0 or miss_cost < 0:
        raise ValueError('Costs must be non-negative.')
    return {'act': action_cost, 'wait': probability * miss_cost}

def choose_action(probability, action_cost, miss_cost):
    losses = expected_losses(probability, action_cost, miss_cost)
    action = min(losses, key=losses.get)
    return {'probability': probability, 'losses': losses, 'action': action}

DECISIONS = [choose_action(probability, action_cost=12, miss_cost=100) for probability in (0.10, 0.40)]
for decision in DECISIONS:
    print(decision)

**Expected loss** is a probability-weighted consequence. Here, waiting has expected loss `P(event) × miss cost`; acting has the stated action cost. The probability does not choose the action by itself. Costs, capacity, and unmodeled consequences remain explicit assumptions.

## Model probabilities and thresholds

**Prediction checkpoint — write before running:** Predict how raising a threshold from 0.50 to 0.95 changes acted-on cases, false positives, and missed events. Which direction should operational loss move when a missed event costs eight times an action?

In [ ]:
def confusion_counts(records, threshold):
    counts = {'true_positive': 0, 'false_positive': 0, 'true_negative': 0, 'false_negative': 0}
    for row in records:
        predicted = int(row['model_probability'] >= threshold)
        key = {
            (1, 1): 'true_positive',
            (1, 0): 'false_positive',
            (0, 0): 'true_negative',
            (0, 1): 'false_negative',
        }[(predicted, row['outcome'])]
        counts[key] += 1
    return counts

def threshold_operational_loss(counts, action_cost=1, miss_cost=8):
    acted_on = counts['true_positive'] + counts['false_positive']
    return acted_on * action_cost + counts['false_negative'] * miss_cost

THRESHOLD_RESULTS = []
for threshold in (0.50, 0.85, 0.95):
    counts = confusion_counts(rows, threshold)
    THRESHOLD_RESULTS.append({
        'threshold': threshold,
        **counts,
        'operational_loss': threshold_operational_loss(counts),
    })
for result in THRESHOLD_RESULTS:
    print(result)

## Calibration intuition after the observations

A model is calibrated for a reference population when cases assigned probability near `p` contain the event about fraction `p` over repeated outcomes. Calibration does not mean every individual event matches its score. It can differ by cohort or time, so a single aggregate can hide deployment shift.

In [ ]:
def brier_score(records):
    if not records:
        raise ValueError('Brier score needs at least one record.')
    return sum((row['model_probability'] - row['outcome']) ** 2 for row in records) / len(records)

stable_rows = [row for row in rows if row['cohort'] == 'stable']
shifted_rows = [row for row in rows if row['cohort'] == 'deployment_shift']
BRIER_BY_COHORT = {
    'stable': brier_score(stable_rows),
    'deployment_shift': brier_score(shifted_rows),
}
print(BRIER_BY_COHORT)

## Code reading

Read `reliability_report` below before executing it. Trace the input schema, grouping key, accumulator mutations, denominator, output order, and empty-input behavior.

**Prediction checkpoint — write before running:** Manually trace the 0.90 group and predict its count, mean probability, observed frequency, and signed gap.

In [ ]:
def reliability_report(records, group_key='model_probability'):
    accumulators = {}
    for row in records:
        key = row[group_key]
        bucket = accumulators.setdefault(key, {'count': 0, 'probability_sum': 0.0, 'outcome_sum': 0})
        bucket['count'] += 1
        bucket['probability_sum'] += row['model_probability']
        bucket['outcome_sum'] += row['outcome']
    report = []
    for key, bucket in sorted(accumulators.items(), key=lambda item: str(item[0])):
        count = bucket['count']
        mean_probability = bucket['probability_sum'] / count
        observed_frequency = bucket['outcome_sum'] / count
        report.append({
            'group': key,
            'count': count,
            'mean_probability': mean_probability,
            'observed_frequency': observed_frequency,
            'calibration_gap': observed_frequency - mean_probability,
        })
    return report

for group in reliability_report(rows):
    print(group)

## Verify the controlled-failure repair

Keep sensitivity and specificity fixed.

**Prediction checkpoint — write before running:** As the base rate changes from 1% to 10% to 50%, predict the direction of `P(condition | positive)`. Explain why sensitivity can stay fixed while the posterior changes.

In [ ]:
def posterior_from_rates(base_rate, sensitivity, specificity):
    true_positive_share = base_rate * sensitivity
    false_positive_share = (1 - base_rate) * (1 - specificity)
    positive_share = true_positive_share + false_positive_share
    if positive_share == 0:
        raise ValueError('No positive results exist under these rates.')
    return true_positive_share / positive_share

BASE_RATE_SWEEP = {
    base_rate: posterior_from_rates(base_rate, sensitivity=0.90, specificity=0.91)
    for base_rate in (0.01, 0.10, 0.50)
}
print(BASE_RATE_SWEEP)

## Executable contract checks

These assertions check the lab machinery, not learner completion. Predict whether each invariant should hold before running.

In [ ]:
assert len(rows) == 40
assert MODEL_EVENT_COUNTS['event'] + MODEL_EVENT_COUNTS['complement'] == MODEL_EVENT_COUNTS['total']
assert sum(SCREENING[key] for key in ('true_positive', 'false_negative', 'false_positive', 'true_negative')) == SCREENING['population']
assert P_POSITIVE_GIVEN_CONDITION == 0.90
assert 0.09 < P_CONDITION_GIVEN_POSITIVE < 0.10
assert DEPENDENCE['relationship'] == 'dependent'
assert choose_action(0.10, 12, 100)['action'] == 'wait'
assert choose_action(0.40, 12, 100)['action'] == 'act'
assert all(0 <= score <= 1 for score in BRIER_BY_COHORT.values())
assert list(BASE_RATE_SWEEP.values()) == sorted(BASE_RATE_SWEEP.values())
print('M17 executable contract checks passed.')

## No-AI transfer and decision record

Close this notebook and complete `missions/M17/no_ai_gate.md` without AI-generated code or prose. Then complete `missions/M17/adr_prompt.md` for the threshold decision. The package deliberately contains no completed learner response or selected threshold.

## Concept map after the experiments

`event definition → repeated reference population → frequency/base rate → condition direction → dependence assumptions → model probability/reliability → consequence model → action`.

When uncertainty reasoning fails, locate the first broken link: an ambiguous event, wrong denominator, neglected base rate, false independence, unstable calibration, or hidden consequence assumption.